# Aula 3 — Análise Exploratória e Limpeza dos Dados

**Intensivão Quant AI — ImpactUFSCar**

---

Antes de calcular qualquer sinal, precisamos entender o que temos. Modelar dados que você não inspecionou é a receita para backtest com resultados enganosos.

Hoje você vai aprender o vocabulário probabilístico que aparece em todo relatório quant — e vai aplicar cada conceito diretamente nos seus dados.

In [ ]:
# ── CONFIGURAÇÃO DO AMBIENTE ─────────────────────────────────────────────────
# Este notebook roda no VS Code (Jupyter local) E no Google Colab.
# Execute esta célula primeiro — ela detecta o ambiente e configura os caminhos.

import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # ── Google Colab ──────────────────────────────────────────────────────────
    print("Ambiente detectado: Google Colab")

    # Montar Google Drive para persistir os dados entre sessões
    from google.colab import drive
    drive.mount('/content/drive')

    # Instalar dependências não incluídas no Colab por padrão
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'yfinance', 'pyarrow'], check=False)

    # Clonar o repositório do curso (pula automaticamente se já existir)
    REPO_DIR = '/content/intensivao-quant-ai'
    if not os.path.exists(REPO_DIR):
        print("Clonando repositório do curso...")
        subprocess.run([
            'git', 'clone',
            'https://github.com/fmaldonadoo/intensivao-quant-ai.git',
            REPO_DIR
        ])
        print("Repositório clonado com sucesso.")

    # Pasta de dados no Google Drive — persiste entre sessões do Colab
    DADOS_DIR = '/content/drive/MyDrive/intensivao_quant/dados'
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados serão salvos em: {DADOS_DIR}")

else:
    # ── VS Code / Jupyter local ───────────────────────────────────────────────
    print("Ambiente detectado: VS Code / Jupyter local")

    # Sobe a árvore de diretórios até encontrar a raiz do repositório (.git)
    _p = os.path.abspath(os.getcwd())
    while _p != os.path.dirname(_p):
        if os.path.exists(os.path.join(_p, '.git')):
            break
        _p = os.path.dirname(_p)

    DADOS_DIR = os.path.join(_p, 'dados')
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados serão salvos em: {DADOS_DIR}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Carregando os dados salvos na Aula 2
precos       = pd.read_parquet(os.path.join(DADOS_DIR, 'precos_ibov.parquet'))
retornos     = pd.read_parquet(os.path.join(DADOS_DIR, 'retornos_diarios.parquet'))
ret_mensais  = pd.read_parquet(os.path.join(DADOS_DIR, 'retornos_mensais.parquet'))

print(f"Preços:          {precos.shape}")
print(f"Retornos diários: {retornos.shape}")
print(f"Retornos mensais: {ret_mensais.shape}")

## 1. Esperança Estatística e Variância nos Mercados Financeiros

Antes de colocarmos qualquer estratégia de trading para rodar, precisamos entender a fundo as ferramentas estatísticas que nos ajudam a medir o comportamento histórico dos ativos. O comportamento de uma ação é uma variável aleatória, e para entendê-la usamos **momentos estatísticos**.

---

### A. Esperança Matemática (Média — $\mu$)
A **esperança** (ou média aritmética) nos dá o "retorno esperado" ou a tendência central dos retornos de um ativo. Ela nos diz qual foi o retorno médio que o ativo entregou por período de tempo.

Matematicamente, para uma amostra de retornos $R_1, R_2, \dots, R_N$:

$$\mu = \mathbb{E}[R] = \frac{1}{N} \sum_{i=1}^N R_i$$

**Intuição para Iniciantes:** Se a média mensal dos retornos de PETR4 nos últimos anos foi de $1.8\%$, a esperança matemática nos diz que, na ausência de outras informações, nossa melhor estimativa de retorno para o próximo mês é $1.8\%$.

---

### B. Variância ($\sigma^2$) e Desvio Padrão (Volatilidade — $\sigma$)
A **variância** mede a dispersão dos retornos em relação à média. Em finanças, ela é a base para a métrica de **risco**. Como a variância está em unidades quadráticas (ex: $\%^2$), nós tiramos a raiz quadrada para obter o **Desvio Padrão**, que no mercado financeiro chamamos de **Volatilidade ($\\sigma$)**.

$$\sigma^2 = \mathbb{E}[(R - \mu)^2] = \frac{1}{N-1} \sum_{i=1}^N (R_i - \mu)^2$$

$$\sigma = \sqrt{\sigma^2}$$

**Intuição para Iniciantes:**
- Um ativo com **baixa volatilidade** (ex: ações de energia elétrica como EGIE3) tem retornos diários muito próximos de sua média. O gráfico é comportado e previsível.
- Um ativo com **alta volatilidade** (ex: ações de tecnologia ou commodities altamente alavancadas como PRIO3) tem retornos que oscilam violentamente longe da média. O potencial de ganho é grande, mas o risco de perdas severas também é elevado.


In [ ]:
# Calculando esperança e volatilidade para todas as ações
media_diaria = retornos.mean()
vol_diaria   = retornos.std()

# Anualização: ~252 dias úteis por ano
# Média anual = média_diaria * 252
# Vol anual   = vol_diaria * sqrt(252)  ← pela regra da raiz do tempo
DIAS_UTEIS = 252

media_anual = media_diaria * DIAS_UTEIS
vol_anual   = vol_diaria * np.sqrt(DIAS_UTEIS)

resumo = pd.DataFrame({'retorno_anual': media_anual, 'vol_anual': vol_anual})
resumo = resumo.dropna().sort_values('retorno_anual', ascending=False)

print("Top 5 maiores retornos anuais médios:")
print(resumo.head().to_string(float_format='{:.2%}'.format))
print()
print("Bottom 5 menores retornos anuais médios:")
print(resumo.tail().to_string(float_format='{:.2%}'.format))

In [ ]:
# Scatter retorno vs risco — o trade-off fundamental
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(resumo['vol_anual'], resumo['retorno_anual'], alpha=0.6, s=50)

# Nomear alguns pontos de interesse
destaques = ['WEGE3.SA', 'PETR4.SA', 'VALE3.SA', 'MGLU3.SA', 'BBAS3.SA']
for ticker in destaques:
    if ticker in resumo.index:
        ax.annotate(
            ticker.replace('.SA', ''),
            (resumo.loc[ticker, 'vol_anual'], resumo.loc[ticker, 'retorno_anual']),
            textcoords='offset points', xytext=(6, 3), fontsize=8
        )

ax.axhline(0, color='red', linewidth=0.8, linestyle='--', label='retorno zero')
ax.set_xlabel('Volatilidade anual')
ax.set_ylabel('Retorno anual médio')
ax.set_title('Risco × Retorno — Ações do IBOVESPA (2010–2024)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend()
plt.tight_layout()
plt.show()

## 2. A Lei dos Grandes Números e o Teorema do Limite Central em Finanças

Para iniciantes em dados quantitativos, é fundamental entender o comportamento das amostras estatísticas à medida que coletamos mais dados.

---

### A. Lei dos Grandes Números (LGN)
A **Lei dos Grandes Números** afirma que, à medida que o tamanho da nossa amostra de dados ($N$) cresce, a média amostral observada se aproxima cada vez mais do verdadeiro valor esperado (a média populacional teórica).

**Intuição Prática:**
- Se você analisar o retorno médio de PETR4 usando apenas **15 dias de dados**, a média pode estar altamente distorcida por um evento atípico ocorrido naquela semana. A sua estimativa é altamente instável e pouco confiável.
- Se você usar **10 anos de dados diários** (cerca de 2500 observações), a média observada será muito mais estável e representativa da verdadeira tendência histórica de geração de retorno da empresa.

---

### B. O Teorema do Limite Central (TLC) e seu limite em Finanças
O **Teorema do Limite Central** nos diz que, se somarmos muitas variáveis aleatórias independentes e identicamente distribuídas, a soma (ou média) dessas variáveis tenderá a uma **distribuição normal (Gaussiana)**, independentemente da distribuição original dos dados individuais.

#### A grande armadilha em Finanças:
Muitos modelos de finanças clássicos assumem que, como os retornos são compostos ao longo do tempo, eles deveriam se comportar de forma Gaussiana (Normal). No entanto, na prática de mercado, **os retornos diários não são perfeitamente independentes nem identicamente distribuídos**.
- O mercado tem memória (fenômeno de agrupamento de volatilidade - *volatility clustering*).
- O mercado reage a notícias de forma não linear (cascatas de ordens, pânico e euforia).
- Por isso, embora o TLC funcione bem para fenômenos físicos simples, os retornos de curtíssimo prazo das ações reais teimam em desobedecer a distribuição normal, exibindo anomalias que estudaremos a seguir.


In [ ]:
# Simulação: lançar uma moeda N vezes e ver a frequência de caras convergir para 0.5
np.random.seed(42)
N = 2000
lancamentos = np.random.randint(0, 2, N)  # 0 = coroa, 1 = cara
media_acumulada = np.cumsum(lancamentos) / np.arange(1, N + 1)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(media_acumulada, label='frequência de caras (acumulada)', color='steelblue')
ax.axhline(0.5, color='red', linestyle='--', label='valor real (0.5)')
ax.set_xlabel('Número de lançamentos')
ax.set_ylabel('Frequência')
ax.set_title('Lei dos Grandes Números — a média amostral converge para a média real')
ax.legend()
plt.tight_layout()
plt.show()

print("Com 10 lançamentos:",  f"{media_acumulada[9]:.2f}")
print("Com 100 lançamentos:", f"{media_acumulada[99]:.2f}")
print("Com 2000 lançamentos:",f"{media_acumulada[-1]:.2f}")

Agora o mesmo princípio aplicado aos nossos dados: a volatilidade estimada de PETR4 com janelas cada vez maiores.

In [ ]:
petr4_ret = retornos['PETR4.SA'].dropna()

# Volatilidade diária calculada com janelas crescentes
tamanhos = range(20, len(petr4_ret), 10)
vols_estimadas = [petr4_ret.iloc[:n].std() * np.sqrt(252) for n in tamanhos]

vol_total = petr4_ret.std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(list(tamanhos), vols_estimadas, color='steelblue')
ax.axhline(vol_total, color='red', linestyle='--',
           label=f'vol com todo o histórico: {vol_total:.1%}')
ax.set_xlabel('Dias de histórico usados')
ax.set_ylabel('Volatilidade anualizada estimada')
ax.set_title('Lei dos Grandes Números — estimativa de volatilidade de PETR4')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend()
plt.tight_layout()
plt.show()

## 3. A Falha da Distribuição Normal: Leptocurtose e "Caudas Pesadas" (Fat Tails)

Se você já fez algum curso básico de estatística, deve se lembrar da famosa **curva de sino (distribuição normal ou Gaussiana)**. Ela é simétrica, elegante e fácil de calcular. Por isso, a finança tradicional (como a fórmula de precificação de opções de Black-Scholes e os modelos de Value-at-Risk mais simples) foi construída assumindo que os retornos dos ativos seguem uma distribuição normal.

No entanto, nos mercados financeiros reais, **a hipótese normal falha gravemente, e essa mentira custa caro!**

---

### Skewness (Assimetria — $S$) e Kurtosis (Curtose — $K$)

Para entender essa falha, olhamos para o terceiro e o quarto momentos estatísticos dos retornos:

#### Assimetria (Skewness):
Mede a falta de simetria da distribuição em relação à média.
- **Normal:** Skewness = 0 (perfeitamente simétrica).
- **Assimetria Negativa ($S < 0$):** A cauda esquerda (perdas) é mais longa e pesada do que a direita. As quedas tendem a ser mais rápidas e extremas do que as altas. **Este é o comportamento típico das ações!** O mercado costuma "subir de escada e descer de elevador".

#### Curtose (Kurtosis):
Mede o peso relativo das caudas e a altura do pico central.
- **Normal:** Curtose = 3.
- **Curtose em Excesso ($K - 3$):** Quando a curtose é maior que 3, chamamos a distribuição de **leptocúrtica**. Isso significa que ela possui caudas muito mais grossas e pesadas (Fat Tails) e um pico central mais alto.

---

### O Perigo das Caudas Pesadas (Fat Tails / Cisnes Negros)

Quando uma distribuição de retornos tem **caudas pesadas**, isso significa que **eventos extremos de mercado (crashes ou super altas) acontecem com uma frequência ordens de grandeza maior do que a distribuição normal prevê.**

Vamos ver uma tabela de probabilidades sob a distribuição normal vs. realidade para variações extremas (medidas em desvios padrões $\sigma$):

| Desvio Padrão ($\sigma$) | Frequência Teórica na Normal | Frequência Real nos Mercados |
|-----------------------|------------------------------|------------------------------|
| $3\sigma$ (movimento forte) | $1$ vez a cada $370$ dias | Ocorre múltiplas vezes por ano |
| $5\sigma$ (crise moderada) | $1$ vez a cada $4.700$ anos | Ocorre em quase toda grande crise de mercado (ex: Joesley Day em 2017) |
| $7\sigma$ (colapso total)  | Praticamente impossível | Ocorre algumas vezes por década (ex: Covid em 2020) |

**Conclusão Quant:** Como os mercados financeiros reais exibem leptocurtose severa, modelos de gestão de risco que assumem normalidade subestimam drasticamente a probabilidade de ruína. É por isso que quants usam métricas de risco robustas como o **Max Drawdown** e o **Sortino Ratio** em vez de confiar cegamente apenas no Sharpe Ratio clássico.


In [ ]:
# Distribuição de retornos de PETR4 vs Normal
petr4_ret = retornos['PETR4.SA'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma com curva normal sobreposta
x = np.linspace(petr4_ret.min(), petr4_ret.max(), 300)
normal_fit = stats.norm.pdf(x, petr4_ret.mean(), petr4_ret.std())

axes[0].hist(petr4_ret, bins=100, density=True, alpha=0.6,
             color='steelblue', label='retornos reais')
axes[0].plot(x, normal_fit, color='red', linewidth=2, label='normal ajustada')
axes[0].set_title('Histograma de retornos diários — PETR4')
axes[0].set_xlabel('Retorno log diário')
axes[0].legend()
axes[0].set_xlim(-0.15, 0.15)

# QQ-plot: se fosse normal, os pontos ficariam na linha
stats.probplot(petr4_ret, dist='norm', plot=axes[1])
axes[1].set_title('QQ-Plot — desvio da normalidade nas caudas')
axes[1].get_lines()[1].set_color('red')

plt.tight_layout()
plt.show()

print(f"Curtose (excess): {petr4_ret.kurtosis():.2f}  (normal = 0)")
print(f"Assimetria:       {petr4_ret.skew():.2f}   (normal = 0)")
print()
print("Eventos extremos observados vs esperado pela normal:")
sigma = petr4_ret.std()
for n in [3, 4, 5]:
    obs   = (petr4_ret.abs() > n * sigma).sum()
    esp   = len(petr4_ret) * 2 * stats.norm.sf(n)
    print(f"  |r| > {n}σ:  observado = {obs:3d} dias  |  esperado pela normal = {esp:.1f} dias")

## 4. Estacionariedade (Stationarity) e o Perigo das Regressões Espúrias

A **estacionariedade** é uma das propriedades estatísticas mais críticas em séries temporais financeiras.

---

### O que significa ser Estacionária?
Uma série temporal é considerada **fracamente estacionária** (ou estacionária de segunda ordem) se suas propriedades estatísticas básicas não mudam ao longo do tempo. Especificamente:
1. **Média constante:** A série oscila ao redor de um valor médio fixo.
2. **Variância constante:** A amplitude de oscilação não explode nem encolhe estruturalmente no tempo.
3. **Autocovariância estável:** A relação de dependência entre duas observações em instantes $t$ e $t-k$ depende apenas do intervalo de tempo (lag $k$) entre elas, não do instante de tempo absoluto $t$.

---

### Por que Preços NÃO são Estacionários ($I(1)$)?
Os preços das ações no longo prazo se comportam como um **passeio aleatório (random walk)**. Eles acumulam uma tendência de alta no longo prazo (devido à inflação, crescimento das empresas, etc.).
- Se calcularmos a média histórica dos preços de PETR4 de 2012 a 2015, ela será completamente diferente da média de 2020 a 2024.
- Séries não-estacionárias contêm "raiz unitária" (denotadas como integrada de ordem 1, ou $I(1)$).
- **O grande perigo:** Se você tentar rodar regressões estatísticas ou modelos de previsão diretamente sobre os preços de dois ativos não-estacionários (ex: preço de PETR4 contra o PIB do Zimbábue), os modelos encontrarão uma relação estatisticamente significante fantástica que **não existe na realidade**. Isso é chamado de **Regressão Espúria**.

---

### Por que Retornos SÃO Estacionários ($I(0)$)?
Quando aplicamos a primeira diferença (a variação percentual ou o log-return dos preços), removemos a tendência e trazemos a série para um estado estacionário ($I(0)$). Os retornos flutuam de forma estável em torno de uma média muito próxima de zero e com variância controlada.

### O Teste Augmented Dickey-Fuller (ADF)
Para testar formalmente se uma série é estacionária, os quants utilizam o teste estatístico **ADF**:
- **Hipótese Nula ($H_0$):** A série possui raiz unitária (não é estacionária).
- **Hipótese Alternativa ($H_1$):** A série é estacionária.
- Se o **p-value** do teste for menor que nosso nível de significância (geralmente $5\%$ ou $1\%$, ou seja, $p < 0.05$), nós rejeitamos $H_0$ e temos evidência estatística sólida de que a série é **estacionária**.


In [ ]:
# Visualização: preço vaga, retorno não
petr4_preco = precos['PETR4.SA'].dropna()

fig, axes = plt.subplots(2, 1, figsize=(13, 7))

# Preço — série não estacionária
petr4_preco.plot(ax=axes[0], color='steelblue')
axes[0].set_title('PETR4 — Preço (não estacionário: não tem nível fixo de retorno)')
axes[0].set_ylabel('Preço (R$)')

# Retorno — série estacionária
petr4_ret.plot(ax=axes[1], color='steelblue', alpha=0.7)
media = petr4_ret.mean()
axes[1].axhline(media, color='red', linewidth=1.2, linestyle='--',
                label=f'média = {media:.4f}')
axes[1].set_title('PETR4 — Retorno diário (estacionário: oscila em torno de média estável)')
axes[1].set_ylabel('Retorno log')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Teste formal de estacionariedade — Augmented Dickey-Fuller
# Hipótese nula (H0): a série TEM raiz unitária (não estacionária)
# Se p-value < 0.05: rejeitamos H0 — a série É estacionária
from statsmodels.tsa.stattools import adfuller

resultado_preco   = adfuller(petr4_preco.dropna())
resultado_retorno = adfuller(petr4_ret.dropna())

print("Teste ADF — PETR4 Preço")
print(f"  p-value: {resultado_preco[1]:.4f}  →  {'Não estacionária ✗' if resultado_preco[1] > 0.05 else 'Estacionária ✓'}")

print()
print("Teste ADF — PETR4 Retorno")
print(f"  p-value: {resultado_retorno[1]:.4f}  →  {'Não estacionária ✗' if resultado_retorno[1] > 0.05 else 'Estacionária ✓'}")

## 5. Autocorrelação: O Mercado tem Memória?

A **autocorrelação** mede o grau de associação linear entre uma série temporal e ela mesma defasada no tempo por um determinado número de períodos (chamados de lags).

---

### A Matemática da Autocorrelação
A autocorrelação de lag $k$ ($
ho_k$) mede a correlação entre a série de retornos $r_t$ e a série de retornos atrasada $r_{t-k}$:

$$\rho_k = \frac{\text{Cov}(r_t, r_{t-k})}{\text{Var}(r_t)}$$

---

### Intuição Prática para Iniciantes
- **Autocorrelação Positiva Forte:** Significa que retornos positivos hoje tendem a ser seguidos por retornos positivos amanhã (um comportamento de **tendência ou momentum** no curto prazo).
- **Autocorrelação Negativa Forte:** Significa que retornos positivos hoje tendem a ser sucedidos por retornos negativos amanhã (comportamento de **reversão à média ou mean-reversion**).
- **Autocorrelação zero:** Os retornos de hoje não fornecem nenhuma pista linear sobre os retornos de amanhã. É a base da Hipótese de Mercado Eficiente (HME) na sua forma fraca.

Na prática de mercado, ações individuais em frequência diária apresentam autocorrelação muito próxima de zero. No entanto, em períodos mensais, certos fatores de estilo (como o **Momentum** que usaremos) exibem correlação positiva de médio prazo, o que nos permite extrair alfa do mercado.


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Autocorrelação nos retornos diários
plot_acf(petr4_ret.dropna(), lags=30, ax=axes[0], alpha=0.05)
axes[0].set_title('Autocorrelação — PETR4 Retornos DIÁRIOS (lags 1–30 dias)')
axes[0].set_xlabel('Lag (dias)')

# Autocorrelação nos retornos mensais
plot_acf(ret_mensais['PETR4.SA'].dropna(), lags=24, ax=axes[1], alpha=0.05)
axes[1].set_title('Autocorrelação — PETR4 Retornos MENSAIS (lags 1–24 meses)')
axes[1].set_xlabel('Lag (meses)')

plt.tight_layout()
plt.show()

print("Autocorrelação de 1 dia  (retornos diários): ",
      f"{petr4_ret.autocorr(lag=1):.4f}")
print("Autocorrelação de 1 mês  (retornos mensais):",
      f"{ret_mensais['PETR4.SA'].autocorr(lag=1):.4f}")
print("Autocorrelação de 12 meses (ret. mensais): ",
      f"{ret_mensais['PETR4.SA'].autocorr(lag=12):.4f}")

## 6. Tratamento de dados ausentes

Na Aula 2 identificamos NaNs em várias ações. Agora decidimos o que fazer com eles.

**Estratégia para o intensivão:**
- Em **retornos**: NaN significa que a ação não tinha preço naquele dia (estava suspensa ou ainda não tinha IPO). Mantemos o NaN — não inventamos retorno.
- Em **preços**: para cálculos de rolling window, usamos `ffill()` (último preço disponível) apenas onde necessário.

O critério de decisão é a **cobertura temporal**: ações com menos de 80% de dias com dados são excluídas.

In [ ]:
# Cobertura de cada ação — % de dias com dado válido
cobertura = retornos.notna().mean().sort_values()

fig, ax = plt.subplots(figsize=(13, 4))
cobertura.plot(kind='bar', ax=ax, color='steelblue', width=0.8)
ax.axhline(0.8, color='red', linestyle='--', linewidth=1.5, label='corte: 80%')
ax.set_title('Cobertura temporal por ação (% de dias com retorno disponível)')
ax.set_xlabel('')
ax.set_ylabel('Cobertura')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend()
ax.set_xticklabels([t.replace('.SA', '') for t in cobertura.index], rotation=90, fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# Filtro de cobertura mínima
COBERTURA_MINIMA = 0.80

tickers_validos = cobertura[cobertura >= COBERTURA_MINIMA].index.tolist()
tickers_removidos = cobertura[cobertura < COBERTURA_MINIMA].index.tolist()

print(f"Tickers mantidos:  {len(tickers_validos)}")
print(f"Tickers removidos: {len(tickers_removidos)}")
if tickers_removidos:
    print(f"  {[t.replace('.SA','') for t in tickers_removidos]}")

## 7. Filtro de liquidez

Por que filtrar ativos ilíquidos?

1. **Custo real:** ações com baixo volume têm spread alto — a diferença entre o preço de compra e de venda pode eliminar o retorno da estratégia
2. **Capacidade:** é impossível executar uma estratégia em ativos que não têm volume suficiente para absorver a sua ordem
3. **Backtest enganoso:** ativos ilíquidos têm preços que às vezes não se movem por dias — o retorno calculado parece baixo, mas na prática você não conseguiria ter comprado/vendido

Proxy de liquidez que usamos: **volume médio diário**. Mantemos apenas ações com volume médio acima de um threshold.

In [ ]:
import yfinance as yf

# Baixamos o volume para os tickers válidos
print("Baixando volume... (pode demorar um momento)")
volume_raw = yf.download(
    tickers_validos,
    start='2010-01-01',
    end='2024-12-31',
    auto_adjust=True,
    progress=False
)['Volume']

# Volume médio diário por ação
volume_medio = volume_raw.mean().sort_values()

# Threshold: mantemos ações acima do percentil 20 de volume
threshold_volume = volume_medio.quantile(0.20)

tickers_liquidos = volume_medio[volume_medio >= threshold_volume].index.tolist()

print(f"\nThreshold de volume (p20): {threshold_volume:,.0f} ações/dia")
print(f"Tickers após filtro de liquidez: {len(tickers_liquidos)}")

## 8. Rolling statistics — preparando terreno para o sinal

Uma **janela rolling** (deslizante) recalcula uma estatística a cada novo dia usando apenas os últimos N dias. É diferente de calcular uma estatística com todo o histórico.

Por que usamos rolling?
- A volatilidade de uma ação muda ao longo do tempo (em crises ela explode, em períodos calmos cai)
- Queremos capturar as condições **recentes**, não as de 10 anos atrás

Na Aula 7 vamos dividir o sinal de momentum pela volatilidade rolling. Por isso vale já visualizar como ela se comporta.

In [ ]:
# Volatilidade rolling de PETR4 com janelas de 21 e 63 dias
petr4_ret = retornos['PETR4.SA'].dropna()

vol_21d  = petr4_ret.rolling(21).std()  * np.sqrt(252)
vol_63d  = petr4_ret.rolling(63).std()  * np.sqrt(252)
vol_252d = petr4_ret.rolling(252).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(13, 5))
vol_21d.plot(ax=ax,  label='vol 21d (1 mês)',   alpha=0.7)
vol_63d.plot(ax=ax,  label='vol 63d (3 meses)',  alpha=0.7)
vol_252d.plot(ax=ax, label='vol 252d (1 ano)',   alpha=0.9, linewidth=2)
ax.set_title('Volatilidade rolling anualizada — PETR4')
ax.set_ylabel('Volatilidade anualizada')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend()
plt.tight_layout()
plt.show()

## 9. Dataset limpo — salvando para as próximas aulas

In [ ]:
# Dataset final: ações que passaram no filtro de cobertura e liquidez
tickers_finais = [t for t in tickers_liquidos if t in tickers_validos]

precos_limpo   = precos[tickers_finais]
retornos_limpo = retornos[tickers_finais]
ret_mens_limpo = ret_mensais[tickers_finais]

print(f"Dataset limpo: {len(tickers_finais)} ações × {len(retornos_limpo)} dias")

precos_limpo.to_parquet(os.path.join(DADOS_DIR, 'precos_limpo.parquet'))
retornos_limpo.to_parquet(os.path.join(DADOS_DIR, 'retornos_diarios_limpo.parquet'))
ret_mens_limpo.to_parquet(os.path.join(DADOS_DIR, 'retornos_mensais_limpo.parquet'))

# Salva a lista de tickers para usar nas próximas aulas
pd.Series(tickers_finais, name='ticker').to_csv('tickers_finais.csv', index=False)

print("\nArquivos salvos:")
print("  precos_limpo.parquet")
print("  retornos_diarios_limpo.parquet")
print("  retornos_mensais_limpo.parquet")
print("  tickers_finais.csv")

## Resumo da aula

| Conceito | O que aprendemos | Impacto na estratégia |
|---|---|---|
| Esperança e variância | Retorno médio e dispersão — os dois números que resumem uma distribuição | Base do Sharpe ratio (aula 5) |
| Lei dos grandes números | Com poucos dados, as estimativas são instáveis | Justifica usar 10+ anos de histórico |
| Fat tails e curtose | Retornos têm caudas mais pesadas que a normal | Drawdowns reais são maiores do que modelos normais preveem |
| Estacionariedade | Preço não é; retorno é | Sempre modele retornos |
| Autocorrelação | Quase zero no diário; momentum é fenômeno de meses | Justifica janela de 12 meses no sinal |
| Filtro de cobertura e liquidez | Removemos ações com histórico curto ou baixo volume | Dataset confiável para o backtest |
| Rolling statistics | Volatilidade muda no tempo — janelas curtas capturam o regime atual | Base do sinal v2 (aula 7) |

---

## Para replicar em casa

1. Rode o notebook do início ao fim e confirme que os 4 arquivos foram salvos
2. Explore: qual ação tem a maior curtose? Qual tem a maior assimetria negativa?
3. Observe o gráfico risco × retorno: algum ponto parece fora do padrão? Por quê?

---

*ImpactUFSCar — Diretoria de Quant*